# Friend A — full training from scratch on all 199 videos

This notebook implements task 3 assigned to Friend A in `TEAM.md` after the leak-free validator run:

- train the complete `TemporalUNet3D + detection head + SimpleNodeTransformer` model jointly, end-to-end;
- initialise every model component from scratch — no pilkwang or team checkpoint;
- use all 199 local training videos (71 `44b6` + 128 `6bba`);
- run 30 full epochs by default on one 16 GB GPU;
- use cosine learning-rate decay from `1e-4` to `1e-5` across the 30 epochs;
- save `edge_predictor_best.pth`, numbered checkpoints every 5 epochs, and `config.json` for Friend B/Vitalii.

Because all 199 labelled videos participate in training, there is no independent local holdout in this stage. The fixed 12-video `44b6` list is reused only as a leaky internal monitoring set required by the trainer; its `acc * recall` must not be reported as LOEO or as an official competition score.

Important:

1. Start Jupyter from the repository root and select the Python 3.12 project environment.
2. Run all cells. Training starts automatically and may take roughly 24–30 hours on the local RTX 5070 Ti.
3. The generated command is rejected if it contains `--init-weights` or `--unet-weights`.
4. Detector and linker are trained jointly by `train_unet_transformer.py`, exactly as prescribed by `TEAM.md`.
5. `--max-detections 1024` limits predicted local maxima per frame before the quadratic transformer; it does not limit sparse GT nodes. Peaks near labelled GT are protected. Set it to `0` for a future dense-label experiment if more than 1024 true cells per frame must be retained.
6. Zarr and Blosc decompression concurrency is bounded and Zarr array handles are reused to avoid host-RAM growth during multi-day Windows runs.
7. Friend B performs prediction and evaluation. Do not submit anything to the leaderboard here.

Optional overrides before starting Jupyter:

```powershell
$env:BIOHUB_DATA_DIR = "C:\path\to\biohub-cell-tracking-during-development\train"
$env:BIOHUB_FULL199_EPOCHS = "30"
$env:BIOHUB_FULL199_MIN_LR = "1e-5"
$env:BIOHUB_FULL199_MAX_DETECTIONS = "1024"  # transformer memory guard
$env:BIOHUB_FULL199_CHECKPOINT_EVERY = "5"
jupyter lab
```


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path


EXPERIMENT = "local_friend_a_exp02_full199_scratch_30ep"
METHOD = "friend_a_full199_scratch_bce_w2"

# Full-training recipe selected after the leak-free validator.
RUN_TRAINING = os.environ.get("BIOHUB_RUN_TRAINING", "1") != "0"
EPOCHS = int(os.environ.get("BIOHUB_FULL199_EPOCHS", "30"))
BATCH_SIZE = int(os.environ.get("BIOHUB_FULL199_BATCH", "4"))
default_workers = "0" if os.name == "nt" else "4"
NUM_WORKERS = int(os.environ.get("BIOHUB_FULL199_WORKERS", default_workers))
LEARNING_RATE = float(os.environ.get("BIOHUB_FULL199_LR", "1e-4"))
LR_SCHEDULE = os.environ.get("BIOHUB_FULL199_LR_SCHEDULE", "cosine")
MIN_LEARNING_RATE = float(os.environ.get("BIOHUB_FULL199_MIN_LR", "1e-5"))
SEED = int(os.environ.get("BIOHUB_FULL199_SEED", "42"))
WINDOW_SIZE = int(os.environ.get("BIOHUB_FULL199_WINDOW", "2"))
DET_LOSS = os.environ.get("BIOHUB_FULL199_DET_LOSS", "bce")
DET_NEG_WEIGHT = float(os.environ.get("BIOHUB_FULL199_DET_NEG_WEIGHT", "0.01"))
DET_LOSS_WEIGHT = float(os.environ.get("BIOHUB_FULL199_DET_LOSS_WEIGHT", "1.0"))
DOWNSAMPLE = os.environ.get("BIOHUB_FULL199_DOWNSAMPLE", "1,4,4")
POOL_KERNEL_UM = float(os.environ.get("BIOHUB_FULL199_POOL_KERNEL_UM", "5.0"))
MAX_DETECTIONS = int(os.environ.get("BIOHUB_FULL199_MAX_DETECTIONS", "1024"))
CHECKPOINT_EVERY = int(os.environ.get("BIOHUB_FULL199_CHECKPOINT_EVERY", "5"))
AUG_INTENSITY = os.environ.get("BIOHUB_FULL199_AUG_INTENSITY", "0") == "1"
MAX_ITERS_TEXT = os.environ.get("BIOHUB_FULL199_MAX_ITERS", "").strip()
MAX_ITERS = int(MAX_ITERS_TEXT) if MAX_ITERS_TEXT else None

if not 30 <= EPOCHS <= 50:
    raise ValueError("TEAM.md requires 30–50 epochs for full training on all 199 videos")
if BATCH_SIZE not in (4, 5, 6):
    raise ValueError("Use batch 4–6 without gradient checkpointing on a 16 GB GPU")
if MAX_DETECTIONS < 0:
    raise ValueError("BIOHUB_FULL199_MAX_DETECTIONS must be non-negative; 0 disables the cap")
if CHECKPOINT_EVERY < 0:
    raise ValueError("BIOHUB_FULL199_CHECKPOINT_EVERY must be non-negative")
if LR_SCHEDULE not in {"constant", "cosine"}:
    raise ValueError("BIOHUB_FULL199_LR_SCHEDULE must be constant or cosine")
if LR_SCHEDULE == "cosine" and not 0 <= MIN_LEARNING_RATE <= LEARNING_RATE:
    raise ValueError("BIOHUB_FULL199_MIN_LR must be between 0 and BIOHUB_FULL199_LR")
if DET_LOSS not in {"bce", "gauss_focal"}:
    raise ValueError("BIOHUB_FULL199_DET_LOSS must be bce or gauss_focal")


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        repo_candidates = [candidate, candidate / ".github-push-worktree"]
        for repo_candidate in repo_candidates:
            if (repo_candidate / "TEAM.md").exists() and (repo_candidate / "tools" / "repo" / "scripts").exists():
                return repo_candidate
    raise FileNotFoundError("Repository root not found. Start Jupyter inside C:\\Projects\\CellTracker.")


PROJECT_ROOT = find_project_root(Path.cwd())

explicit_data = os.environ.get("BIOHUB_DATA_DIR", "").strip()
data_candidates = [
    Path(explicit_data).expanduser() if explicit_data else None,
    PROJECT_ROOT / "biohub-cell-tracking-during-development" / "train",
    PROJECT_ROOT / "train",
    PROJECT_ROOT.parent / "train",
]
DATA_DIR = next((path.resolve() for path in data_candidates if path is not None and path.exists()), None)
if DATA_DIR is None:
    checked = "\n".join(str(path) for path in data_candidates if path is not None)
    raise FileNotFoundError(
        "Training data were not found. Set BIOHUB_DATA_DIR to the directory containing *.zarr and *.geff.\n"
        f"Checked:\n{checked}"
    )

RUN_ID = os.environ.get("BIOHUB_RUN_ID", datetime.now().strftime("%Y%m%d_%H%M%S"))
default_output_root = (
    PROJECT_ROOT.parent / "outputs" if PROJECT_ROOT.name == ".github-push-worktree" else PROJECT_ROOT / "outputs"
)
OUTPUT_ROOT = Path(os.environ.get("BIOHUB_OUTPUT_DIR", default_output_root)).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / EXPERIMENT / RUN_ID
TRAIN_REPO = RUN_DIR / "train_repo"
ARTIFACT_DIR = RUN_DIR / "artifact"
LOEO_MANIFEST = PROJECT_ROOT / "experiments" / "_lib" / "loeo_train6bba_val44b6.json"
PATCH_SOURCE = PROJECT_ROOT / "experiments" / "_lib" / "train_patches.py"

print(f"project: {PROJECT_ROOT}")
print(f"data:    {DATA_DIR}")
print(f"run:     {RUN_DIR}")
print(
    f"recipe: scratch end-to-end on all 199 | epochs={EPOCHS} batch={BATCH_SIZE} workers={NUM_WORKERS} "
    f"lr={LEARNING_RATE:g}->{MIN_LEARNING_RATE:g} ({LR_SCHEDULE}) window={WINDOW_SIZE} det_loss={DET_LOSS} "
    f"max_detections={MAX_DETECTIONS} checkpoint_every={CHECKPOINT_EVERY} seed={SEED}"
)


In [ ]:
# Preflight, immutable split conversion, and a writable copy of the training code.

required_modules = ["torch", "tracksdata", "geff", "polars", "rustworkx", "zarr", "tqdm"]
evaluation_modules = ["scipy", "skimage", "dask", "networkx", "pandas", "sklearn", "pyscipopt", "ilpy", "blosc2"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError(
        "Missing packages: " + ", ".join(missing_modules) + "\n"
        "Install the dependencies listed in TEAM.md into the selected Python 3.12 environment."
    )
missing_evaluation_modules = [name for name in evaluation_modules if importlib.util.find_spec(name) is None]
if missing_evaluation_modules:
    print(
        "Warning: optional inference/evaluation packages are missing: "
        + ", ".join(missing_evaluation_modules)
        + ". They do not block training, but TEAM.md requires them before official evaluation."
    )

import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Python 3.12 is required by TEAM.md; current interpreter: {sys.version.split()[0]}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Install a CUDA 12.8+ PyTorch build and restart the notebook kernel.")

# A real allocation catches driver/build mismatches that nvidia-smi alone cannot detect.
probe = torch.zeros(1, device="cuda")
del probe
torch.cuda.empty_cache()
device_name = torch.cuda.get_device_name(0)
device_capability = torch.cuda.get_device_capability(0)
print(
    f"CUDA OK: torch={torch.__version__}, runtime={torch.version.cuda}, "
    f"device={device_name}, capability={device_capability}"
)

loeo_manifest = json.loads(LOEO_MANIFEST.read_text(encoding="utf-8"))
monitor_ids = list(loeo_manifest["holdout"])
train_ids = sorted(
    path.name.removesuffix(".zarr")
    for path in DATA_DIR.glob("*.zarr")
    if (DATA_DIR / f"{path.name.removesuffix('.zarr')}.geff").exists()
)

prefix_counts = {prefix: sum(dataset_id.startswith(prefix + "_") for dataset_id in train_ids) for prefix in ("44b6", "6bba")}
if len(train_ids) != 199 or prefix_counts != {"44b6": 71, "6bba": 128}:
    raise RuntimeError(f"Expected all 199 videos (71 44b6 + 128 6bba), found {len(train_ids)}: {prefix_counts}")
if len(train_ids) != len(set(train_ids)):
    raise RuntimeError("Duplicate training dataset IDs")
if len(monitor_ids) != 12 or not set(monitor_ids).issubset(train_ids):
    raise RuntimeError("The fixed 12-video monitoring set must be present inside the all-199 training set")

missing_data = []
for dataset_id in train_ids:
    for suffix in (".zarr", ".geff"):
        expected = DATA_DIR / f"{dataset_id}{suffix}"
        if not expected.exists():
            missing_data.append(str(expected))
if missing_data:
    preview = "\n".join(missing_data[:20])
    raise FileNotFoundError(f"Missing {len(missing_data)} dataset paths. First entries:\n{preview}")

print(f"Full-training set OK: {len(train_ids)} videos | {prefix_counts}")
print(f"Internal monitor: {len(monitor_ids)} videos, all also included in training (not independent)")

RUN_DIR.mkdir(parents=True, exist_ok=False)
TRAIN_REPO.mkdir(parents=True)
for subdir in ("scripts", "src"):
    shutil.copytree(PROJECT_ROOT / "tools" / "repo" / subdir, TRAIN_REPO / subdir)

patch_spec = importlib.util.spec_from_file_location("friend_a_train_patches", PATCH_SOURCE)
if patch_spec is None or patch_spec.loader is None:
    raise RuntimeError(f"Cannot import training patches from {PATCH_SOURCE}")
patch_module = importlib.util.module_from_spec(patch_spec)
patch_spec.loader.exec_module(patch_module)
TRAIN_SCRIPT = TRAIN_REPO / "scripts" / "train_unet_transformer.py"
patch_module.apply_train_patches(TRAIN_SCRIPT)
trainer_source = TRAIN_SCRIPT.read_text(encoding="utf-8")
legacy_models_import = "from biohub_tracking.models import SimpleNodeTransformer, TemporalUNet3D"
if legacy_models_import in trainer_source:
    trainer_source = trainer_source.replace(
        legacy_models_import,
        "from biohub_tracking.models.simple_node_transformer import SimpleNodeTransformer\n"
        "from biohub_tracking.models.temporal_unet import TemporalUNet3D",
        1,
    )
    TRAIN_SCRIPT.write_text(trainer_source, encoding="utf-8")
    print("patched local namespace-package model imports")
compile(TRAIN_SCRIPT.read_text(encoding="utf-8"), str(TRAIN_SCRIPT), "exec")

help_env = {
    **os.environ,
    "PYTHONPATH": str(TRAIN_REPO / "src"),
    "BIOHUB_DATA_DIR": str(DATA_DIR),
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
}
help_check = subprocess.run(
    [sys.executable, str(TRAIN_SCRIPT), "--help"],
    cwd=TRAIN_REPO,
    env=help_env,
    text=True,
    encoding="utf-8",
    errors="replace",
    capture_output=True,
)
if help_check.returncode != 0:
    raise RuntimeError("Patched trainer import/CLI check failed:\n" + (help_check.stdout + help_check.stderr)[-4000:])
for required_flag in ("--amp", "--no-grad-ckpt", "--seed", "--det-loss", "--max-detections", "--lr-schedule", "--min-lr", "--checkpoint-every"):
    if required_flag not in help_check.stdout:
        raise RuntimeError(f"Patched trainer is missing required flag: {required_flag}")

# The trainer requires a test list to select a checkpoint. These 12 videos are deliberately
# also in train_ids; this is monitoring only and must never be described as independent validation.
split_for_trainer = [{"split": 0, "train": train_ids, "test": monitor_ids}]
TRAIN_SPLIT = RUN_DIR / "full199_train_with_leaky_monitor12.json"
TRAIN_SPLIT.write_text(json.dumps(split_for_trainer, ensure_ascii=False, indent=2), encoding="utf-8")

TRAIN_CMD = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--method", METHOD,
    "--data-dir", str(DATA_DIR),
    "--splits", str(TRAIN_SPLIT),
    "--split", "0",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LEARNING_RATE),
    "--lr-schedule", LR_SCHEDULE,
    "--min-lr", str(MIN_LEARNING_RATE),
    "--det-loss-weight", str(DET_LOSS_WEIGHT),
    "--det-neg-weight", str(DET_NEG_WEIGHT),
    "--window-size", str(WINDOW_SIZE),
    "--downsample", DOWNSAMPLE,
    "--pool-kernel-um", str(POOL_KERNEL_UM),
    "--max-detections", str(MAX_DETECTIONS),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--det-loss", DET_LOSS,
    "--seed", str(SEED),
    "--amp",
    "--no-grad-ckpt",
    "--single-gpu",
]
if MAX_ITERS is not None:
    TRAIN_CMD.extend(["--max-iters", str(MAX_ITERS)])
if AUG_INTENSITY:
    TRAIN_CMD.append("--aug-intensity")

for forbidden_flag in ("--init-weights", "--unet-weights"):
    if forbidden_flag in TRAIN_CMD:
        raise RuntimeError(f"Leakage guard: forbidden scratch-training flag found: {forbidden_flag}")

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    git_commit = None

RUN_CONFIG = {
    "experiment": EXPERIMENT,
    "run_id": RUN_ID,
    "status": "prepared",
    "created_at": datetime.now().astimezone().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "git_commit": git_commit,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_device": device_name,
    "train_scope": "all_199_videos",
    "train_embryos": ["44b6", "6bba"],
    "train_videos": train_ids,
    "monitoring_videos": monitor_ids,
    "monitoring_is_independent": False,
    "monitor_manifest_sha256": hashlib.sha256(LOEO_MANIFEST.read_bytes()).hexdigest(),
    "initialization": "scratch",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "learning_rate_schedule": LR_SCHEDULE,
    "minimum_learning_rate": MIN_LEARNING_RATE,
    "seed": SEED,
    "window_size": WINDOW_SIZE,
    "det_loss": DET_LOSS,
    "det_neg_weight": DET_NEG_WEIGHT,
    "det_loss_weight": DET_LOSS_WEIGHT,
    "downsample": DOWNSAMPLE,
    "pool_kernel_um": POOL_KERNEL_UM,
    "max_detections_per_frame": MAX_DETECTIONS,
    "checkpoint_every_n_epochs": CHECKPOINT_EVERY,
    "aug_intensity": AUG_INTENSITY,
    "max_iters_per_epoch": MAX_ITERS,
    "amp": True,
    "gradient_checkpointing": False,
    "command": TRAIN_CMD,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "command.txt").write_text(subprocess.list2cmdline(TRAIN_CMD) + "\n", encoding="utf-8")
(RUN_DIR / "results.json").write_text(
    json.dumps(
        {
            "experiment": EXPERIMENT,
            "run_id": RUN_ID,
            "status": "prepared",
            "official_metric": None,
            "note": "All 199 videos are training data; per-epoch monitor metrics are leaky diagnostics. Official prediction/evaluation is Friend B's follow-up task.",
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print("scratch command prepared (no pretrained weights):")
print(subprocess.list2cmdline(TRAIN_CMD))


In [ ]:
# Stream the long-running process to the notebook and to train.log.

import csv
import math
import re


def write_results(payload: dict) -> None:
    (RUN_DIR / "results.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def terminate_process(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)


def print_subprocess_line(line: str) -> None:
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    safe_line = line.encode(encoding, errors="replace").decode(encoding, errors="replace")
    print(safe_line, end="", flush=True)


def write_training_history(epoch_rows: list[dict], output_dir: Path) -> tuple[Path, Path | None]:
    """Save per-epoch CSV and a dependency-free SVG diagnostic plot."""
    history_csv = output_dir / "history.csv"
    fields = ["epoch", "lr", "edge", "det", "test_loss", "acc", "recall", "best"]
    with history_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows([{**row, "epoch": row["epoch"] + 1} for row in epoch_rows])
    if not epoch_rows:
        return history_csv, None

    width, height = 960, 680
    left, right = 95.0, 900.0
    loss_top, loss_bottom = 105.0, 325.0
    metric_top, metric_bottom = 420.0, 620.0
    epochs = [row["epoch"] + 1 for row in epoch_rows]

    def x_pos(index: int) -> float:
        return (left + right) / 2 if len(epoch_rows) == 1 else left + index * (right - left) / (len(epoch_rows) - 1)

    positive_losses = [
        float(row[key]) for row in epoch_rows for key in ("lr", "edge", "det", "test_loss") if float(row[key]) > 0
    ]
    log_low = math.floor(math.log10(min(positive_losses)))
    log_high = math.ceil(math.log10(max(positive_losses)))
    if log_low == log_high:
        log_low -= 1

    def loss_y(value: float) -> float:
        value = max(float(value), 10 ** log_low)
        ratio = (log_high - math.log10(value)) / (log_high - log_low)
        return loss_top + ratio * (loss_bottom - loss_top)

    metric_values = [float(row[key]) for row in epoch_rows for key in ("recall", "best")]
    metric_low = max(0.0, min(metric_values) - 0.01)
    metric_high = min(1.0, max(metric_values) + 0.01)
    if metric_high - metric_low < 0.02:
        metric_low = max(0.0, metric_low - 0.01)
        metric_high = min(1.0, metric_high + 0.01)

    def metric_y(value: float) -> float:
        ratio = (metric_high - float(value)) / (metric_high - metric_low)
        return metric_top + ratio * (metric_bottom - metric_top)

    svg = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<style>text{font-family:Segoe UI,Arial,sans-serif;fill:#202124}.axis{stroke:#5f6368;stroke-width:1}.grid{stroke:#dfe3e8;stroke-width:1}.series{fill:none;stroke-width:3}.dot{stroke:#fff;stroke-width:1.5}</style>',
        '<text x="480" y="34" text-anchor="middle" font-size="21" font-weight="600">Friend A full-199 scratch training</text>',
        '<text x="480" y="58" text-anchor="middle" font-size="14" fill="#5f6368">Leaky internal monitoring — not independent and not the official competition metric</text>',
        '<text x="95" y="88" font-size="16" font-weight="600">Losses and learning rate (log scale)</text>',
        '<text x="95" y="403" font-size="16" font-weight="600">Internal monitoring metrics</text>',
    ]

    for exponent in range(log_high, log_low - 1, -1):
        y = loss_y(10 ** exponent)
        svg.extend([
            f'<line class="grid" x1="{left}" y1="{y:.1f}" x2="{right}" y2="{y:.1f}"/>',
            f'<text x="{left - 12}" y="{y + 5:.1f}" text-anchor="end" font-size="12">1e{exponent}</text>',
        ])
    for tick_index in range(5):
        value = metric_low + tick_index * (metric_high - metric_low) / 4
        y = metric_y(value)
        svg.extend([
            f'<line class="grid" x1="{left}" y1="{y:.1f}" x2="{right}" y2="{y:.1f}"/>',
            f'<text x="{left - 12}" y="{y + 5:.1f}" text-anchor="end" font-size="12">{value:.3f}</text>',
        ])
    svg.extend([
        f'<line class="axis" x1="{left}" y1="{loss_top}" x2="{left}" y2="{loss_bottom}"/>',
        f'<line class="axis" x1="{left}" y1="{loss_bottom}" x2="{right}" y2="{loss_bottom}"/>',
        f'<line class="axis" x1="{left}" y1="{metric_top}" x2="{left}" y2="{metric_bottom}"/>',
        f'<line class="axis" x1="{left}" y1="{metric_bottom}" x2="{right}" y2="{metric_bottom}"/>',
    ])

    loss_series = [("det", "Detection loss", "#d97706"), ("edge", "Edge loss", "#2563eb"), ("test_loss", "Validation loss", "#7c3aed"), ("lr", "Learning rate", "#64748b")]
    for key, label, colour in loss_series:
        points = " ".join(f"{x_pos(i):.1f},{loss_y(row[key]):.1f}" for i, row in enumerate(epoch_rows))
        svg.append(f'<polyline class="series" points="{points}" stroke="{colour}"/>')
        for i, row in enumerate(epoch_rows):
            svg.append(f'<circle class="dot" cx="{x_pos(i):.1f}" cy="{loss_y(row[key]):.1f}" r="4" fill="{colour}"/>')

    metric_series = [("recall", "Validation recall", "#059669"), ("best", "Best internal score", "#dc2626")]
    for key, label, colour in metric_series:
        points = " ".join(f"{x_pos(i):.1f},{metric_y(row[key]):.1f}" for i, row in enumerate(epoch_rows))
        svg.append(f'<polyline class="series" points="{points}" stroke="{colour}"/>')
        for i, row in enumerate(epoch_rows):
            svg.append(f'<circle class="dot" cx="{x_pos(i):.1f}" cy="{metric_y(row[key]):.1f}" r="4" fill="{colour}"/>')

    best_index = max(range(len(epoch_rows)), key=lambda index: epoch_rows[index]["best"])
    best_x = x_pos(best_index)
    svg.extend([
        f'<line x1="{best_x:.1f}" y1="{metric_top}" x2="{best_x:.1f}" y2="{metric_bottom}" stroke="#111827" stroke-dasharray="5 5"/>',
        f'<text x="{best_x + 7:.1f}" y="{metric_top + 18:.1f}" font-size="12">saved checkpoint</text>',
    ])
    for i, epoch in enumerate(epochs):
        x = x_pos(i)
        svg.append(f'<text x="{x:.1f}" y="{metric_bottom + 25}" text-anchor="middle" font-size="12">{epoch}</text>')
    svg.append(f'<text x="{(left + right) / 2:.1f}" y="{metric_bottom + 52}" text-anchor="middle" font-size="14">Epoch</text>')

    legend_items = [*loss_series, *metric_series]
    for i, (_, label, colour) in enumerate(legend_items):
        x = 105 + (i % 3) * 260
        y = 348 + (i // 3) * 24
        svg.extend([
            f'<line x1="{x}" y1="{y}" x2="{x + 28}" y2="{y}" stroke="{colour}" stroke-width="3"/>',
            f'<text x="{x + 36}" y="{y + 5}" font-size="13">{label}</text>',
        ])
    svg.append('</svg>')
    curve_svg = output_dir / "training_curves.svg"
    curve_svg.write_text("\n".join(svg), encoding="utf-8")
    return history_csv, curve_svg


if RUN_TRAINING:
    started_at = datetime.now().astimezone()
    train_log = RUN_DIR / "train.log"
    trainer_env = {
        **os.environ,
        "BIOHUB_DATA_DIR": str(DATA_DIR),
        "PYTHONPATH": str(TRAIN_REPO / "src"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    }
    print(f"Training started at {started_at.isoformat()}", flush=True)
    with train_log.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            TRAIN_CMD,
            cwd=TRAIN_REPO,
            env=trainer_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                print_subprocess_line(line)
            return_code = process.wait()
        except KeyboardInterrupt:
            terminate_process(process)
            write_results(
                {
                    "experiment": EXPERIMENT,
                    "run_id": RUN_ID,
                    "status": "interrupted",
                    "official_metric": None,
                    "log": str(train_log),
                }
            )
            raise
        except BaseException:
            terminate_process(process)
            write_results(
                {
                    "experiment": EXPERIMENT,
                    "run_id": RUN_ID,
                    "status": "notebook_stream_failed",
                    "official_metric": None,
                    "log": str(train_log),
                }
            )
            raise

    finished_at = datetime.now().astimezone()
    wall_seconds = (finished_at - started_at).total_seconds()
    if return_code != 0:
        write_results(
            {
                "experiment": EXPERIMENT,
                "run_id": RUN_ID,
                "status": "failed",
                "return_code": return_code,
                "official_metric": None,
                "wall_seconds": wall_seconds,
                "log": str(train_log),
            }
        )
        raise RuntimeError(f"Training failed with exit code {return_code}; see {train_log}")

    checkpoint_source = TRAIN_REPO / "weights" / METHOD / "split_0" / "edge_predictor_best.pth"
    model_config_source = checkpoint_source.parent / "config.json"
    if not checkpoint_source.exists() or not model_config_source.exists():
        raise FileNotFoundError("Training ended successfully but the expected checkpoint/config pair is missing")

    ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
    checkpoint_target = ARTIFACT_DIR / "edge_predictor_best.pth"
    model_config_target = ARTIFACT_DIR / "config.json"
    shutil.copy2(checkpoint_source, checkpoint_target)
    shutil.copy2(model_config_source, model_config_target)
    periodic_checkpoint_sources = sorted(checkpoint_source.parent.glob("edge_predictor_epoch_*.pth"))
    periodic_checkpoint_targets = []
    for periodic_source in periodic_checkpoint_sources:
        periodic_target = ARTIFACT_DIR / periodic_source.name
        shutil.copy2(periodic_source, periodic_target)
        periodic_checkpoint_targets.append(periodic_target)
    shutil.copy2(RUN_DIR / "run_config.json", ARTIFACT_DIR / "run_config.json")

    log_text = train_log.read_text(encoding="utf-8", errors="replace")
    epoch_pattern = re.compile(
        r"Epoch\s+(?P<epoch>\d+)/(?P<epochs>\d+).*?lr=(?P<lr>[0-9.eE+-]+).*?edge=(?P<edge>[0-9.eE+-]+).*?"
        r"det=(?P<det>[0-9.eE+-]+).*?test_loss=(?P<test_loss>[0-9.eE+-]+).*?"
        r"acc=(?P<acc>[0-9.eE+-]+).*?recall=(?P<recall>[0-9.eE+-]+).*?best=(?P<best>[0-9.eE+-]+)"
    )
    epoch_rows = []
    for match in epoch_pattern.finditer(log_text):
        row = {"epoch": int(match.group("epoch"))}
        row.update({key: float(match.group(key)) for key in ("lr", "edge", "det", "test_loss", "acc", "recall", "best")})
        epoch_rows.append(row)

    checkpoint_sha256 = hashlib.sha256(checkpoint_target.read_bytes()).hexdigest()
    history_csv, curve_svg = write_training_history(epoch_rows, ARTIFACT_DIR)
    results = {
        "experiment": EXPERIMENT,
        "run_id": RUN_ID,
        "status": "trained_not_officially_scored",
        "started_at": started_at.isoformat(),
        "finished_at": finished_at.isoformat(),
        "wall_seconds": wall_seconds,
        "train_videos": len(train_ids),
        "monitoring_videos": len(monitor_ids),
        "monitoring_set": "fixed 12-video 44b6 subset also included in training",
        "monitoring_is_independent": False,
        "initialization": "scratch",
        "checkpoint": str(checkpoint_target),
        "checkpoint_sha256": checkpoint_sha256,
        "periodic_checkpoints": [str(path) for path in periodic_checkpoint_targets],
        "model_config": str(model_config_target),
        "history_csv": str(history_csv),
        "training_curve_svg": str(curve_svg) if curve_svg is not None else None,
        "internal_sparse_validation": epoch_rows,
        "official_metric": None,
        "note": "Internal acc/recall is leaky diagnostic only because all 199 videos are in training; Friend B must run prediction and the official metric.",
    }
    write_results(results)
    shutil.copy2(RUN_DIR / "results.json", ARTIFACT_DIR / "results.json")
    print(f"Training complete in {wall_seconds / 3600:.2f} h")
    print(f"Hand-off artifact: {ARTIFACT_DIR}")
else:
    print("BIOHUB_RUN_TRAINING=0: preflight passed; training was intentionally skipped.")


In [ ]:
# Verify and describe the exact hand-off to Friend B/Vitalii. This cell never uploads or submits anything.

results_path = RUN_DIR / "results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))

if results["status"] == "trained_not_officially_scored":
    required_artifacts = [
        ARTIFACT_DIR / "edge_predictor_best.pth",
        ARTIFACT_DIR / "config.json",
        ARTIFACT_DIR / "run_config.json",
        ARTIFACT_DIR / "results.json",
        ARTIFACT_DIR / "history.csv",
        ARTIFACT_DIR / "training_curves.svg",
    ]
    missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
    if missing_artifacts:
        raise FileNotFoundError("Incomplete hand-off artifact:\n" + "\n".join(missing_artifacts))

    handoff = {
        "experiment": EXPERIMENT,
        "status": "ready_for_friend_b_evaluation",
        "weights_dataset_subfolder": EXPERIMENT,
        "files_for_weights_dataset": [
            str(ARTIFACT_DIR / "edge_predictor_best.pth"),
            str(ARTIFACT_DIR / "config.json"),
            *[str(path) for path in sorted(ARTIFACT_DIR.glob("edge_predictor_epoch_*.pth"))],
        ],
        "supporting_files": [
            str(ARTIFACT_DIR / "run_config.json"),
            str(ARTIFACT_DIR / "results.json"),
            str(ARTIFACT_DIR / "history.csv"),
            str(ARTIFACT_DIR / "training_curves.svg"),
            str(RUN_DIR / "train.log"),
        ],
        "evaluation_request": {
            "sets": ["48-video regression diagnostic (leaky for full-199 model)", "competition test when authorised"],
            "metric": "official competition metric",
            "leaderboard_submission": False,
        },
    }
    handoff_path = RUN_DIR / "handoff.json"
    handoff_path.write_text(json.dumps(handoff, ensure_ascii=False, indent=2), encoding="utf-8")
    shutil.copy2(handoff_path, ARTIFACT_DIR / "handoff.json")
    print("READY FOR HAND-OFF")
    print(f"Send this directory to Friend B/Vitalii: {ARTIFACT_DIR}")
    print(f"Then copy the measured result into experiments/{EXPERIMENT}/results.json and RESULTS.md.")
else:
    print(f"Current run status: {results['status']}. No checkpoint hand-off was created.")
